In [ ]:
# !pip install yfinance requests transformers torch xgboost scikit-learn pandas plotly

In [1]:
import yfinance as yf         # API Yahoo Finance 
import requests               # Pour interroger l'API REST HTTP de NewsAPI
import pandas as pd       
import numpy as np           
from datetime import datetime, timedelta # Dates
from transformers import pipeline 
import xgboost as xgb             
from sklearn.ensemble import RandomForestClassifier 

print("Tous les modules ont été importés avec succès ! L'environnement est prêt.")

Tous les modules ont été importés avec succès ! L'environnement est prêt.


## I. Test Extraction Yahoo Finance

In [3]:
# 1. Définir le symbole boursier (ticker) de l'entreprise (ex: Apple)
ticker = "AAPL"
action = yf.Ticker(ticker)

# 2. Récupérer l'historique des prix sur le dernier mois (1 month)
print(f"Téléchargement des données pour {ticker}...")
df_prix = action.history(period="1mo")

# 3. Filtrer pour garder uniquement les colonnes requises par le projet
colonnes_requises = ["Open", "High", "Low", "Close", "Volume"]
df_prix = df_prix[colonnes_requises]

# 4. Afficher les 5 premières lignes pour vérifier que ça marche
df_prix.head()

Téléchargement des données pour AAPL...


,Open,High,Low,Close,Volume
Date,,,,,
2026-07-22 00:00:00-04:00,327.587466,328.716497,323.061371,325.609192,38755900
2026-07-23 00:00:00-04:00,321.452790,323.021414,319.074836,321.382843,40840800
2026-07-24 00:00:00-04:00,321.512728,334.081875,321.342861,332.733032,47489400
2026-07-27 00:00:00-04:00,334.251737,339.277401,333.732166,336.619690,49604300
2026-07-28 00:00:00-04:00,339.736982,342.594533,335.310806,339.786926,51859000


## II. Test Extraction NewsAPI

In [ ]:
# API key for NewsAPI (You need to register on newsapi.org to get a free key)
NEWS_API_KEY = "6bef262549ab4b92a72ab2642be1d7c0" 

def fetch_financial_news(query: str, days_back: int = 7) -> pd.DataFrame:
    """
    Fetches recent news articles related to a specific company or ticker.
    
    Args:
        query (str): The search term (e.g., "Apple" or "AAPL").
        days_back (int): Number of days to look back for news.
        
    Returns:
        pd.DataFrame: A dataframe containing the publication date, title, and summary of the articles.
    """
    # Calculate dates for the API request using datetime and timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    
    # Format dates as strings (YYYY-MM-DD) required by NewsAPI
    from_param = start_date.strftime('%Y-%m-%d')
    to_param = end_date.strftime('%Y-%m-%d')
    
    # NewsAPI endpoint for searching all articles
    url = "https://newsapi.org/v2/everything"
    
    # Parameters for the API request
    params = {
        'q': query,
        'from': from_param,
        'to': to_param,
        'language': 'en',  # English news is mandatory for FinBERT compatibility
        'sortBy': 'relevancy',
        'apiKey': NEWS_API_KEY
    }
    
    # Send the HTTP GET request to the API
    response = requests.get(url, params=params)
    
    # Check if the request was successful (HTTP status code 200)
    if response.status_code == 200:
        data = response.json()
        articles = data.get('articles', [])
        
        # Extract only the relevant fields (Headline and Summary) for our NLP analysis
        extracted_data = []
        for item in articles:
            # Check if title and description exist to avoid NoneType errors
            if item.get('title') and item.get('description'):
                extracted_data.append({
                    'Date': item['publishedAt'][:10], # Keep only the YYYY-MM-DD part
                    'Headline': item['title'],
                    'Summary': item['description']
                })
            
        # Convert the list of dictionaries into a Pandas DataFrame
        df_news = pd.DataFrame(extracted_data)
        return df_news
    else:
        print(f"Error fetching news: {response.status_code} - {response.text}")
        return pd.DataFrame()

# --- Test the news extraction ---
df_apple_news = fetch_financial_news(query="Apple OR AAPL", days_back=7)

# Display the first 5 rows to visualize our text dataset
df_apple_news.head()

         Date                                           Headline  \
0  2026-08-21  ChatGPT’s Mac App Can Now Control iMessage, Wh...   
1  2026-08-18    Apple squashes EU beef with new App Store rules   
2  2026-08-15  Zyzz Was the Original Looksmaxxer. His Rise an...   
3  2026-08-17  Apple ordered to change app data consent promp...   
4  2026-08-18  Unearthed Video Seems to Reveal Apple AirPods ...   

                                             Summary  
0  Even if Apple didn’t watch iMessage like a haw...  
1  Apple is once again overhauling App Store rule...  
2  A new Apple documentary explores the life of A...  
3  Apple's changing its rules for data collection...  
4  Is this the quiet—and presumably accidental—un...  
